### Stopwords for Refined Analysis

I think that removing [stopwords](https://www.geeksforgeeks.org/nlp/removing-stop-words-nltk-python/) would be helpful for my analysis. I want to find unique words in the poems – the ones that really constitute the meaning in the poem – and the words that are common in the poems are probably unnecessary.

#### Two Approaches:

There are two ways I can think of to go about this cleaning. One approach is just to use a pre-defined stopwords dictionary (or maybe combining multiple pre-defined ones). Two common ones I found are NLTK and spaCy, so I will try those.

The other potential option is just creating some logical rule to extract words that is specific to my dataset. For instance, maybe there are words that are common in poetry but not as common in everyday language. I could define a rule like:

&nbsp;&nbsp;&nbsp;&nbsp;IF *# of times a word appears in total dataset* IS GREATER THAN *# of poems in the dataset*:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;THEN *exclude it from the dataset*

This would eliminate words that, *on average*, appear more than once per poem (so are less likely to be unique to certain kinds of poetry). This plan is based on nothing other than simple logic, though, and I think it might be safer to go with a pre-defined dictionary.

<hr>

### Approach 1: spaCy (because it's [faster?](https://medium.com/@prabhuss73/spacy-vs-nltk-a-comprehensive-comparison-of-two-popular-nlp-libraries-in-python-b66dc477a689))

In [7]:
import pandas as pd
import spacy

In [8]:
df = pd.read_csv('data/df_long.csv')

In [12]:
en = spacy.load("en_core_web_sm")
stopwords = en.Defaults.stop_words
print(stopwords)

{'perhaps', 'must', 'amount', 'former', '’ll', 'seem', 'ever', 'empty', 'what', 'several', 'as', 'throughout', 'sixty', 'used', 'whereupon', 'if', '’m', 'twenty', 'bottom', 'anyway', 'say', 'everything', 'using', 'herein', 'yourself', 'anywhere', 'first', 'only', 'has', 'towards', "'ve", 'thence', 'with', 'you', 'therein', 'others', 'this', 'into', 'may', 'due', '’re', 'one', 'they', 'at', 'have', 'when', 'there', 'myself', 'been', 'get', 'except', 'alone', 'sometime', 'upon', 'thereby', 'such', 'ourselves', 'while', 'did', 'could', 'fifty', 'amongst', 'become', 'regarding', 'being', 'everyone', "'s", 'back', 'but', 'call', 'which', 'in', 'beforehand', 'somehow', 'to', "'m", 'sometimes', 'afterwards', 'by', 'during', 'besides', 'does', 'neither', 'least', 'between', 'above', 'two', 'meanwhile', 'had', 'below', 'full', 'whoever', 'do', "'ll", 'for', 'whither', 'otherwise', 'hundred', 'eleven', 'very', 'take', 'show', 'move', 'be', 'them', 'an', 'more', 'though', 'him', 'next', 'few', 'm

In [16]:
stopwords_list = list(stopwords)

Before I do my stopwords filtering, my dataframe is about 3.2 million rows long

In [17]:
df.shape

(3217195, 6)

I originally was following this [youtube tutorial](https://www.youtube.com/watch?v=vUPAOU2NPls) that has me doing the stopword analysis as though I have a big paragraph of text that I'm trying to remove stopwords from. In reality, my words are already separated into individual columns. I think I can just define a list of words that is the spaCy stopwords library, then filter my df so that it only contains rows where the "word" column is not contained in the stopwords list

In [ ]:
### My original attempt from the youtube tutorial:


# def preprocess(text):
#     doc = nlp(text)
    
#     no_stop_words = [token.text for token in doc if not token.is_stop]
#     return " ".join(no_stop_words)   

In [23]:
df_filtered_spacy = df[~df['word'].isin(stopwords_list)]
df_filtered_spacy.head()

,Title,Poet,Year,URL,position,word
3,The New Testament,Lewis Warsh,15,https://poets.org/poem/new-testament,3,loincloth
4,The New Testament,Lewis Warsh,15,https://poets.org/poem/new-testament,4,covers
6,The New Testament,Lewis Warsh,15,https://poets.org/poem/new-testament,6,lower
10,The New Testament,Lewis Warsh,15,https://poets.org/poem/new-testament,10,body
15,The New Testament,Lewis Warsh,15,https://poets.org/poem/new-testament,15,telling


In [28]:
df_filtered_spacy.shape

(1569876, 6)

After I filter with the spaCy list, I have about 1.5 million rows (more than 50% reduction!)

But now my `position` column is messed up because I removed some of the words :(

I'll have to do something like group by title and poet, sort by position, then reassign position to the words that are there

The below code to reassign position values so they follow 1-step increments was **written by Claude** and it is an interesting approach that I like

In [41]:
## Filter the df so that it is grouped by title and poet and the order of the words (relative to each other in the orig poem) is preserved
df_filtered_spacy = df_filtered_spacy.sort_values(['Title', 'Poet', 'position'])

## Use .cumcount() to county each duplicate of the iterations of Title and Poet
df_filtered_spacy['position_new'] = df_filtered_spacy.groupby(['Title', 'Poet']).cumcount().astype('Int64')+1

In [42]:
df_filtered_test.head(50)

,Title,Poet,Year,URL,position,word,position_new
2141844,"""Ah, Bleak and Barren Was the Moor""",William Makepeace Thackeray,2018,https://poets.org/poem/ah-bleak-and-barren-was...,0,ah,1
2141845,"""Ah, Bleak and Barren Was the Moor""",William Makepeace Thackeray,2018,https://poets.org/poem/ah-bleak-and-barren-was...,1,bleak,2
2141847,"""Ah, Bleak and Barren Was the Moor""",William Makepeace Thackeray,2018,https://poets.org/poem/ah-bleak-and-barren-was...,3,barren,3
2141850,"""Ah, Bleak and Barren Was the Moor""",William Makepeace Thackeray,2018,https://poets.org/poem/ah-bleak-and-barren-was...,6,moor,4
2141851,"""Ah, Bleak and Barren Was the Moor""",William Makepeace Thackeray,2018,https://poets.org/poem/ah-bleak-and-barren-was...,7,ah,5
2141852,"""Ah, Bleak and Barren Was the Moor""",William Makepeace Thackeray,2018,https://poets.org/poem/ah-bleak-and-barren-was...,8,loud,6
2141854,"""Ah, Bleak and Barren Was the Moor""",William Makepeace Thackeray,2018,https://poets.org/poem/ah-bleak-and-barren-was...,10,piercing,7
2141857,"""Ah, Bleak and Barren Was the Moor""",William Makepeace Thackeray,2018,https://poets.org/poem/ah-bleak-and-barren-was...,13,storm,8
2141859,"""Ah, Bleak and Barren Was the Moor""",William Makepeace Thackeray,2018,https://poets.org/poem/ah-bleak-and-barren-was...,15,cottage,9
2141860,"""Ah, Bleak and Barren Was the Moor""",William Makepeace Thackeray,2018,https://poets.org/poem/ah-bleak-and-barren-was...,16,roof,10
